# Figure S18

Compares precipitation forcing among the three response classes.


In [ ]:
from pathlib import Path
import warnings

import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings('ignore', category=RuntimeWarning)


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise FileNotFoundError('Could not find outputs/RECON_MAIN_2011_2023.')


def display_path(path: Path) -> str:
    return str(path.resolve().relative_to(ROOT))


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
LABELS_PATH = RECON / 'metrics' / 'clustering' / 'cluster_labels.csv'
DYNAMIC_PATH = ROOT / 'data' / 'train_val_test_inputs' / 'GNN_spacetime' / 'H6' / 'dynamic_monthly_cache.pt'
PRECIP_DIR = ROOT / 'data' / '10 precipitation' / 'daymet_prcp_monthly_v4r1_mrva'
PRECIP_MATRIX_PATH = PRECIP_DIR / 'daymet_prcp_monthly_mrva_1km.npy'
PRECIP_MONTH_PATH = PRECIP_DIR / 'month_index.csv'
PRECIP_GRID_PATH = PRECIP_DIR / 'grid_lookup.csv'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS18'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CLIMATE_FIG_PATH = OUT_DIR / 'FigS18_ef_climate_forcing.png'

CLASS_ORDER = ['Fast recovery', 'Slow recovery', 'Buffered']
CLASS_COLORS = {
    'Fast recovery': '#2D5FB8',
    'Slow recovery': '#C44E72',
    'Buffered': '#13A8A2',
}
EXPORT_DPI = 600

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9.0,
    'axes.labelsize': 9.5,
    'axes.titlesize': 10.0,
    'xtick.labelsize': 8.3,
    'ytick.labelsize': 8.3,
    'legend.fontsize': 8.2,
    'axes.linewidth': 0.75,
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': EXPORT_DPI,
    'savefig.bbox': 'tight',
})

In [ ]:
labels = pd.read_csv(LABELS_PATH)
required = {
    'grid_id', 'row', 'col', 'x', 'y', 'valid_for_clustering', 'response_class',
}
missing = required.difference(labels.columns)
if missing:
    raise KeyError(f'Missing required label columns: {sorted(missing)}')

labels['valid_for_clustering'] = labels['valid_for_clustering'].astype(bool)
valid = labels['valid_for_clustering'].to_numpy() & labels['response_class'].notna().to_numpy()

cache = torch.load(DYNAMIC_PATH, map_location='cpu', weights_only=False)
if not np.array_equal(np.asarray(cache['grid_ids']), labels['grid_id'].to_numpy()):
    raise ValueError('Dynamic cache and class labels are not aligned by grid_id.')

precipitation_grid = pd.read_csv(PRECIP_GRID_PATH)
precipitation_months = pd.read_csv(PRECIP_MONTH_PATH)['month_label'].astype(str).to_numpy()
precipitation_mm = np.asarray(
    np.load(PRECIP_MATRIX_PATH, mmap_mode='r'), dtype=np.float32
).T
if not np.array_equal(
    np.asarray(cache['grid_ids']), precipitation_grid['grid_id'].to_numpy()
):
    raise ValueError('Precipitation grid and dynamic cache are not aligned by grid_id.')
if not np.array_equal(
    np.asarray(cache['month_labels']).astype(str), precipitation_months
):
    raise ValueError('Precipitation months and dynamic cache are not aligned.')
if precipitation_mm.shape != np.asarray(cache['data']).shape[:2]:
    raise ValueError('Unexpected precipitation matrix shape.')

month_labels = np.asarray(cache['month_labels']).astype(str)
years = np.asarray([int(label[:4]) for label in month_labels])
months = np.asarray([int(label[5:7]) for label in month_labels])
precipitation_stage_masks = {
    'Pre-drought\n(Jan–Apr 2012)': (
        (years == 2012) & (months >= 1) & (months <= 4)
    ),
    'Drought\n(May–Oct 2012)': (
        (years == 2012) & (months >= 5) & (months <= 10)
    ),
    'Early recovery\n(Nov 2012–Apr 2013)': (
        ((years == 2012) & (months >= 11))
        | ((years == 2013) & (months <= 4))
    ),
}

print(f'Valid classified cells: {valid.sum():,}')
print(f'Precipitation matrix: {precipitation_mm.shape}')

In [ ]:
def style_axis(ax: plt.Axes, grid_axis: str | None = None) -> None:
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.75)
        spine.set_color('#333333')
    ax.tick_params(length=3.0, width=0.7, direction='out')
    if grid_axis is not None:
        ax.grid(axis=grid_axis, color='#E5E5E5', linewidth=0.45, zorder=0)
        ax.set_axisbelow(True)


stage_items = list(precipitation_stage_masks.items())
total_precipitation = np.nansum(precipitation_mm, axis=1)
plot_items = [('Total\n(2011–2023)', total_precipitation)]
plot_items.extend((stage_label, np.nansum(precipitation_mm[:, stage_mask], axis=1)) for stage_label, stage_mask in stage_items)
fig_climate, climate_axes = plt.subplots(1, 4, figsize=(13.2, 3.8), facecolor='white')
fig_climate.subplots_adjust(left=0.065, right=0.99, bottom=0.20, top=0.84, wspace=0.22)
stage_ymax = 1.06 * np.nanpercentile(np.concatenate([values for _, values in plot_items[1:]]), 99.5)
for plot_idx, (axis, (title, total_values)) in enumerate(zip(climate_axes, plot_items)):
    box_data = [
        total_values[valid & (response_class == class_name)]
        for class_name in CLASS_ORDER
    ]
    box = axis.boxplot(
        box_data, positions=np.arange(1, len(CLASS_ORDER) + 1), widths=0.52,
        whis=(5, 95), showfliers=False, patch_artist=True,
        medianprops={'color': '#202020', 'linewidth': 1.15},
        whiskerprops={'color': '#444444', 'linewidth': 0.75},
        capprops={'color': '#444444', 'linewidth': 0.75},
        boxprops={'linewidth': 0.75},
    )
    for patch, class_name in zip(box['boxes'], CLASS_ORDER):
        patch.set_facecolor(CLASS_COLORS[class_name])
        patch.set_alpha(0.78)
        patch.set_edgecolor('#333333')
    axis.set_title(title, loc='left', fontweight='bold', pad=5)
    axis.set_ylabel('Precipitation (mm)' if plot_idx == 0 else '')
    axis.set_xlabel('Response class')
    axis.set_xticks(np.arange(1, len(CLASS_ORDER) + 1))
    axis.set_xticklabels(CLASS_ORDER)
    for tick, class_name in zip(axis.get_xticklabels(), CLASS_ORDER):
        tick.set_color(CLASS_COLORS[class_name])
        tick.set_fontweight('bold')
    if plot_idx == 0:
        axis.set_ylim(0, 1.06 * np.nanpercentile(total_values, 99.5))
    else:
        axis.set_ylim(0, stage_ymax)
    style_axis(axis, 'y')
fig_climate.savefig(CLIMATE_FIG_PATH, dpi=EXPORT_DPI, facecolor='white')
plt.show()

print('Saved:', display_path(CLIMATE_FIG_PATH))